# Alternate Probe L1 demonstration

This notebook demonstrates how to generate **flight-like Roman CGI L1 datasets for the
Alternate Probe experiment**, using externally supplied **Gaussian DM probe arrays**
instead of the analytically generated sinusoidal satellite-spot pattern.

It is the Alternate Probe counterpart to `Satellite_spots_demo.ipynb`, and it reuses the
same observing template and the same `corgisim` machinery. Nothing about the L1 file
format, header conventions, or detector representation is redefined here.

**What this notebook does**

1. Explains the observing template and how one custom probe maps onto it.
2. Shows the `add_satspot` / `remove_satspot` mechanism for a custom DM pattern.
3. Calls the committed campaign driver `alt_probe_generate_L1_sims.py` to produce L1 files.
4. Inspects the resulting products: organization by `VISITID`, state ordering, `SATSPOTS`,
   shape/dtype, timestamps, and probe provenance.
5. Displays a correctly located detector crop using cosmic-ray-safe scaling.

> **Status.** Infrastructure validation is complete. The **scientific probe-scale / normalized-intensity
> study is still pending** (see the configuration cell). This notebook uses `scale = 1.0` purely as an
> explicit provisional demonstration value; it is **not** an approved science default.

## Setup before Run All

Complete the setup below **before using Run All**. If you set the environment variables from inside the notebook, run that setup cell before the configuration cell.

### Environment and kernel

Select a Python environment that already provides `corgisim`, `corgidrp`, `proper` and
`roman_preflight_proper`, then pick that environment's existing kernel from the notebook's
kernel picker. This notebook installs no packages and neither registers nor modifies any
Jupyter kernel.

### Data locations

Generated products and the delivered probe arrays live outside the git repositories, so both
locations default to a generic home-directory path and are overridden with plain environment
variables. There is no configuration file and no extra dependency involved.

| Variable | Default | Contents |
|----------|---------|----------|
| `ALT_PROBE_DATA_ROOT` | `~/corgisim-data/alternate-probe` | generated L1 campaigns, and any existing campaign you want to inspect |
| `ALT_PROBE_PROBE_DIR` | `$ALT_PROBE_DATA_ROOT/probes` | the delivered `dmrel_*.fits` Gaussian probe arrays |

Set them in the shell before launching Jupyter:

```bash
export ALT_PROBE_DATA_ROOT=/path/to/alternate-probe-data
export ALT_PROBE_PROBE_DIR=/path/to/probe-delivery
jupyter lab
```

or from inside the notebook, in a cell run **before** the configuration cell below:

```python
import os
os.environ['ALT_PROBE_DATA_ROOT'] = '/path/to/alternate-probe-data'
os.environ['ALT_PROBE_PROBE_DIR'] = '/path/to/probe-delivery'
```

These variables are read **once**, by the configuration cell below, so exporting them after that
cell has run has no effect. Alternatively, edit `DATA_ROOT` / `PROBE_DIR` directly in that cell.

The configuration cell raises `FileNotFoundError` immediately if any probe file is missing, so a
wrong location is reported at once instead of failing later in the campaign.


In [ ]:
import os
import glob
import importlib.util
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

import proper
import roman_preflight_proper

from corgisim import instrument

# This cell only imports packages and loads the committed campaign driver. It does
# NOT read ALT_PROBE_DATA_ROOT or ALT_PROBE_PROBE_DIR: those environment variables
# are consumed by the configuration cell below, which is the single place where the
# data locations are resolved.

# NOTE: roman_preflight_proper.copy_here() is deliberately NOT called here. The
# mechanism demonstration below only touches the DM and needs no PROPER
# prescription in the working directory, and the campaign driver calls
# copy_here() itself when generation is actually requested.

# Locate the repository's examples/ directory whether this notebook is run from
# examples/ or from the repository root, then import the committed campaign driver.
_here = Path.cwd()
EXAMPLES_DIR = _here if (_here / 'alt_probe_generate_L1_sims.py').exists() else _here / 'examples'
DRIVER_PATH = EXAMPLES_DIR / 'alt_probe_generate_L1_sims.py'
assert DRIVER_PATH.exists(), f'campaign driver not found at {DRIVER_PATH}'

_spec = importlib.util.spec_from_file_location('alt_probe_driver', DRIVER_PATH)
driver = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(driver)

print('driver loaded from:', DRIVER_PATH)

### Configuration

Everything that a user is expected to change lives in the single code cell below.

#### The `scale` parameter

`scale` is **required and has no default anywhere** in the library or the driver. Two
conflicting conventions exist and they differ in applied probe intensity by roughly a factor
of 11, so silently picking one would materially change the simulated data:

| `scale` | Meaning |
|---------|---------|
| `1.0`   | Apply the delivered relative-DM array unmodified. |
| `0.3`   | The legacy `corgihowfsc` convention (`GettingProbes.py::get_dm_probes`, `dm1 = dm10 + scale*dmrel`, default `+/-0.3`). A generic default of the HOWFSC probing helper that may not apply to this delivery. |

> #### PROVISIONAL PARAMETERS -- NOT APPROVED SCIENCE VALUES
> `SCALE = 1.0` below is an **explicit provisional demonstration value only**. Choosing the
> correct value requires measuring the resulting probe normalized intensity against the probe's
> design target. **That scientific validation is still outstanding** and is deferred pending
> discussion with Iva and Lukas. No production scale is recommended by this notebook.
>
> The same provisional caveat applies to `BANDPASS`/`COR_TYPE`, `VISTYPE`, the exposure and
> detector settings, and the base DM solution. They are infrastructure-test parameters chosen
> so the pipeline runs end to end; they are **not** confirmed observing parameters.

In [ ]:
# ---------------------------------------------------------------------------
# Data locations
# ---------------------------------------------------------------------------
# Persistent generated products live outside the git repositories. The defaults
# below are generic home-directory paths; override them with the environment
# variables documented in the Setup section above.
DEFAULT_DATA_ROOT = Path.home() / 'corgisim-data' / 'alternate-probe'
DATA_ROOT = Path(os.environ.get(
    'ALT_PROBE_DATA_ROOT', str(DEFAULT_DATA_ROOT)
)).expanduser()

# Delivered Gaussian probe arrays (48x48 relative-DM maps, volts).
# These are read-only source data and are kept outside this repository.
PROBE_DIR = Path(os.environ.get(
    'ALT_PROBE_PROBE_DIR', str(DATA_ROOT / 'probes')
)).expanduser()

PROBE_FILES = [
    PROBE_DIR / 'dmrel_nfov_band1_360deg_ni5e-07_x13_y8_gauss0.fits',
    PROBE_DIR / 'dmrel_nfov_band1_360deg_ni5e-07_x12_y8_gauss1.fits',
    PROBE_DIR / 'dmrel_nfov_band1_360deg_ni5e-07_x13_y7_gauss2.fits',
]

# Fail immediately, and with an actionable message, if the probe location is wrong.
missing_probes = [path for path in PROBE_FILES if not path.is_file()]
if missing_probes:
    missing = "\n".join(f"  - {path}" for path in missing_probes)
    raise FileNotFoundError(
        "Required probe FITS files were not found:\n"
        f"{missing}\n"
        "Set ALT_PROBE_PROBE_DIR before starting Jupyter, "
        "or edit PROBE_DIR in the configuration cell."
    )
print(f'probe files       : {len(PROBE_FILES)} found under {PROBE_DIR}')

# ---------------------------------------------------------------------------
# Probe amplitude -- REQUIRED, PROVISIONAL (see the Configuration notes above)
# ---------------------------------------------------------------------------
SCALE = 1.0          # PROVISIONAL demonstration value, NOT an approved science default.

# ---------------------------------------------------------------------------
# Base DM solution (unprobed dark hole).  PROVISIONAL.
# Leave DM1_PATH/DM2_PATH as None to use the roman_preflight_proper example
# identified by DM_ROOTNAME, or set them to arbitrary DM FITS files.
# ---------------------------------------------------------------------------
DM_ROOTNAME = 'hlc_ni_3e-8'
DM1_PATH = None
DM2_PATH = None

# ---------------------------------------------------------------------------
# Optical configuration.  PROVISIONAL / UNCONFIRMED.
# ---------------------------------------------------------------------------
CGI_MODE = 'excam'
COR_TYPE = 'hlc_band1'
BANDPASS = '1B'                 # PROVISIONAL: '1B' vs '1F' not yet confirmed
OUTPUT_DIM = 153                # 153x153 GITL crop geometry
LOC_X, LOC_Y = 512, 512         # stamp centre within the 1024x1024 science area

# ---------------------------------------------------------------------------
# Observation / detector settings.  PROVISIONAL.
# ---------------------------------------------------------------------------
VISTYPE = 'CGIVST_TDD_OBS'      # PROVISIONAL: correct visit type not yet confirmed
EXPTIME_S = 10.0
N_FRAMES_PER_STATE = 1          # frames per unprobed/positive/negative state
EM_GAIN = 1000.0
PHOTON_COUNTING = False

# Frame spacing. The driver requires frame_time_step_s >= max(exptime, 1.0) so that
# exposures do not overlap and the whole-second SCTSRT stays strictly increasing.
FRAME_TIME_STEP_S = 30.0

# ---------------------------------------------------------------------------
# Output / inspection
# ---------------------------------------------------------------------------
OUTPUT_DIR = DATA_ROOT / 'todo6_demo'

# Generating a campaign is expensive (a full PROPER propagation per DM state).
# Leave GENERATE = False to inspect an existing campaign instead of recomputing it.
#
# NOTE: with GENERATE = True the driver refuses to clobber existing products, so
# OUTPUT_DIR must not already contain the planned output files. Either point
# OUTPUT_DIR at an empty directory or set OVERWRITE = True below. OVERWRITE is
# False by default on purpose: it is the safe setting, and it prevents a partial
# or mistaken re-run from silently destroying an earlier campaign.
GENERATE = False
OVERWRITE = False
EXISTING_L1_DIR = DATA_ROOT / 'todo5_validation'
INSPECT_DIR = OUTPUT_DIR if GENERATE else EXISTING_L1_DIR

print('scale             :', SCALE, '  <-- PROVISIONAL, not an approved science value')
print('output directory  :', OUTPUT_DIR)
print('inspecting        :', INSPECT_DIR)
print('generate campaign :', GENERATE)

## 1. The observing template

The Alternate Probe measurements are acquired with the **satellite-spot imaging observing
template**. The terminology is a little confusing: in normal satellite-spot imaging a
*sinusoidal* pattern is applied to the DM to create bright calibration spots. The Alternate
Probe experiment reuses that same observing sequence, but applies an externally supplied
**Gaussian probe** array instead.

The template consists of three DM states:

| # | State    | DM                            | `SATSPOTS` |
|---|----------|-------------------------------|------------|
| 1 | unprobed | base (dark-hole) solution     | `0`        |
| 2 | positive | base `+ scale * probe`        | `1`        |
| 3 | negative | base `- scale * probe`        | `1`        |

followed by a **restore** step that returns the DM to the pristine base solution.

**One custom pattern is handled per template execution.** A single execution produces exactly
one `unprobed -> positive -> negative -> restore` sequence for one probe file. Therefore:

```
3 probe files  ==  3 separate template executions
               ==  3 distinct VISITIDs
               ==  9 L1 frames (at 1 frame per state)
```

This is **not** one shared unprobed set followed by six probed states. Each probe gets its own
complete trio, so each probe execution is self-contained and independently reducible.

Within a `VISITID`, acquisition order is recoverable from strictly ascending `FTIMEUTC` /
`SCTSRT` values: unprobed frames first, then positive, then negative.

> **Phase 2 note.** `corgidrp`'s `l3_to_l4.find_star` expects satellite-spot data as *three
> equal-sized groups that are all tagged* `SATSPOTS=1`. That differs from the `0/1` split the
> existing `corgisim` machinery produces here. Reconciling the two is a downstream recipe
> concern and is deliberately **not** worked around in the simulation.

## 2. How a custom pattern reaches the DM

Custom probes travel through **exactly the same** `CorgiOptics` entry point as analytical
satellite spots. There is no separate or parallel DM-injection mechanism:

* `optics.add_satspot(satspot_keywords={'custom_pattern': ..., 'scale': ..., 'sign': ..., 'pattern_name': ...})`
  applies `base +/- scale * pattern` to `dm1_v` via `sat_spots.add_custom_pattern_dm`, sets
  `SATSPOTS = 1`, and records provenance in `optics.satspot_info`.
* `optics.remove_satspot(...)` re-enters `add_satspot` with the **inverted sign**, restoring the
  DM, resetting `SATSPOTS = 0` and clearing the provenance.

Passing analytical keywords (`sep_lamD`, `contrast`, ...) together with `custom_pattern` is
rejected, and `scale` is mandatory. The analytical satellite-spot behaviour is unchanged.

The cell below exercises this on a real `CorgiOptics` object. It only touches the DM — no
propagation is performed, so it runs in a fraction of a second.

In [ ]:
dm1_path = DM1_PATH or (roman_preflight_proper.lib_dir + '/examples/' + DM_ROOTNAME + '_dm1_v.fits')
dm2_path = DM2_PATH or (roman_preflight_proper.lib_dir + '/examples/' + DM_ROOTNAME + '_dm2_v.fits')
dm1 = proper.prop_fits_read(dm1_path)
dm2 = proper.prop_fits_read(dm2_path)

optics_keywords = {
    'cor_type': COR_TYPE, 'use_errors': 1, 'polaxis': 10, 'output_dim': OUTPUT_DIM,
    'use_dm1': 1, 'dm1_v': dm1, 'use_dm2': 1, 'dm2_v': dm2,
    'use_fpm': 1, 'use_lyot_stop': 1, 'use_field_stop': 1,
}
optics = instrument.CorgiOptics(CGI_MODE, BANDPASS, optics_keywords=optics_keywords, if_quiet=True)

probe = driver.load_probe_pattern(str(PROBE_FILES[0]))
label = driver.probe_label(str(PROBE_FILES[0]))
dm_base = optics.optics_keywords['dm1_v'].copy()

def dm_state(tag):
    delta = optics.optics_keywords['dm1_v'] - dm_base
    print(f'{tag:<22} SATSPOTS={optics.SATSPOTS}  max(dm-base)={delta.max():+8.4f} V  '
          f'min={delta.min():+8.4f} V  provenance={optics.satspot_info}')

print(f'probe {label!r}: shape={probe.shape}, peak={probe.max():.4f} V')
print()
dm_state('base (unprobed)')

kw = {'custom_pattern': probe, 'scale': SCALE, 'pattern_name': label}
optics.add_satspot(satspot_keywords=kw)                       # positive
dm_state('after +probe')

optics.remove_satspot(satspot_keywords=kw)                    # back to base
kw['sign'] = 'negative'
optics.add_satspot(satspot_keywords=kw)                       # negative
dm_state('after -probe')

optics.remove_satspot(satspot_keywords=kw)                    # restore
dm_state('after restore')
print()
print('DM restored bit-for-bit:', np.array_equal(optics.optics_keywords['dm1_v'], dm_base))

## 3. Generating the L1 campaign

L1 generation is performed by the committed driver `alt_probe_generate_L1_sims.py`, which is a
direct scripted repetition of the satellite-spot template. The campaign logic is **not**
duplicated in this notebook — the driver is the single implementation, and it is also usable
from the command line:

```bash
python alt_probe_generate_L1_sims.py \
    --scale 1.0 \
    --probe-files probe0.fits probe1.fits probe2.fits \
    --n-frames-per-state 1 \
    --output-dir /path/to/data/todo6_demo
```

The driver validates all inputs, every probe, every `VISITID` and every planned output path
*before* writing anything, so an invalid configuration produces no partial output.

Set `GENERATE = True` in the configuration cell to run it. With `N_FRAMES_PER_STATE = 1` and
three probes this performs 9 full PROPER propagations and takes several minutes.

In [ ]:
if GENERATE:
    out_dir = driver.run_campaign(
        scale=SCALE,
        probe_files=[str(p) for p in PROBE_FILES],
        n_frames_per_state=N_FRAMES_PER_STATE,
        exptime=EXPTIME_S,
        em_gain=EM_GAIN,
        photon_counting=PHOTON_COUNTING,
        output_dim=OUTPUT_DIM,
        loc_x=LOC_X,
        loc_y=LOC_Y,
        bandpass=BANDPASS,
        cor_type=COR_TYPE,
        cgi_mode=CGI_MODE,
        dm_rootname=DM_ROOTNAME,
        dm1_path=DM1_PATH,
        dm2_path=DM2_PATH,
        vistype=VISTYPE,
        frame_time_step_s=FRAME_TIME_STEP_S,
        output_dir=str(OUTPUT_DIR),
        overwrite=OVERWRITE,
    )
    print('campaign written to', out_dir)
else:
    print('GENERATE is False -- skipping generation and inspecting the existing campaign at:')
    print('   ', INSPECT_DIR)

## 4. Inspecting the generated L1 products

### 4.1 Organization by `VISITID`

Each probe execution writes into its own `V<VISITID>/` directory. The driver assigns a distinct
`VISITID` per probe by incrementing the observation number, so the three template executions are
cleanly separated on disk.

In [ ]:
visit_dirs = sorted(d for d in glob.glob(str(Path(INSPECT_DIR) / 'V*')) if os.path.isdir(d))
l1_files = []
for vd in visit_dirs:
    files = sorted(glob.glob(os.path.join(vd, '*_l1_.fits')))
    l1_files.extend(files)
    print(f'{os.path.basename(vd)}  ({len(files)} frames)')
    for f in files:
        print('    ', os.path.basename(f))

print()
print(f'{len(visit_dirs)} visit directories, {len(l1_files)} L1 frames total')

### 4.2 State ordering, `SATSPOTS`, format, timestamps and provenance

Probe provenance is written by the existing machinery as `COMMENT` cards in the L1 **primary**
header (`satspot_pattern_name`, `satspot_scale`, `satspot_sign`); `SATSPOTS`, `FTIMEUTC` and
`SCTSRT` live in the **extension** header.

> **Known asymmetry.** `remove_satspot` clears the provenance, so **unprobed frames carry
> `SATSPOTS=0` and no probe COMMENTs at all**. They are associated with their probe execution
> indirectly, through the shared `VISITID` and the ascending acquisition timestamps. Attaching
> explicit provenance to unprobed frames would require changing `instrument.py` and is out of
> scope here.

In [ ]:
def read_l1_summary(path):
    """Return a compact summary dict for one L1 file (no bulk data retained)."""
    with fits.open(path, memmap=False) as hdul:
        pri, ext = hdul[0].header, hdul[1].header
        shape, dtype = hdul[1].data.shape, hdul[1].data.dtype
    prov = {}
    for card in pri.get('COMMENT', []):
        text = str(card)
        if text.startswith('satspot_') and ' : ' in text:
            key, value = text.split(' : ', 1)
            prov[key.strip()] = value.strip()
    return {
        'file': os.path.basename(path),
        'visitid': pri['VISITID'],
        'satspots': int(ext['SATSPOTS']),
        'shape': shape,
        'dtype': str(dtype),
        'ftimeutc': ext['FTIMEUTC'],
        'sctsrt': ext['SCTSRT'],
        'exptime': ext['EXPTIME'],
        'vistype': pri['VISTYPE'],
        'pattern': prov.get('satspot_pattern_name', '-'),
        'scale': prov.get('satspot_scale', '-'),
        'sign': prov.get('satspot_sign', 'unprobed'),
    }

summaries = [read_l1_summary(f) for f in l1_files]

hdr = f"{'VISITID':<20}{'SATSPOTS':>9}{'sign':>10}{'scale':>7}  {'SCTSRT':<21}{'shape':<14}{'dtype':<9}pattern"
print(hdr)
print('-' * len(hdr))
for s in summaries:
    print(f"{s['visitid']:<20}{s['satspots']:>9}{s['sign']:>10}{s['scale']:>7}  "
          f"{s['sctsrt']:<21}{str(s['shape']):<14}{s['dtype']:<9}{s['pattern']}")

In [ ]:
# Consistency checks on the campaign as a whole.
ok = True
def check(condition, message):
    global ok
    print(('  PASS  ' if condition else '  FAIL  ') + message)
    ok = ok and bool(condition)

n_states = 3
check(all(s['shape'] == (1200, 2200) for s in summaries), 'every frame is full-frame 1200x2200')
check(all(s['dtype'] == 'uint16' for s in summaries),     'every frame is uint16')

visitids = [s['visitid'] for s in summaries]
check(len(set(visitids)) == len(visit_dirs), f'{len(set(visitids))} distinct VISITIDs, one per probe execution')
check(all(len(v) == 19 and v.isdigit() for v in visitids), 'every VISITID is a 19-digit numeric string')

times = [s['sctsrt'] for s in summaries]
check(times == sorted(times) and len(set(times)) == len(times),
      'SCTSRT is strictly ascending across the whole campaign')

per_state = N_FRAMES_PER_STATE
expected_flags = ([0] * per_state + [1] * per_state + [1] * per_state) * len(visit_dirs)
check([s['satspots'] for s in summaries] == expected_flags,
      'SATSPOTS follows unprobed=0, positive=1, negative=1 for every trio')

expected_signs = (['unprobed'] * per_state + ['positive'] * per_state + ['negative'] * per_state) * len(visit_dirs)
check([s['sign'] for s in summaries] == expected_signs, 'state ordering is unprobed -> positive -> negative')

probed = [s for s in summaries if s['sign'] != 'unprobed']
check(all(s['scale'] == str(SCALE) for s in probed), f'every probed frame records the explicit scale {SCALE}')
check(len({s['pattern'] for s in probed}) == len(visit_dirs), 'each execution records its own probe label')
check(all(s['pattern'] == '-' for s in summaries if s['sign'] == 'unprobed'),
      'unprobed frames carry no probe provenance (documented asymmetry)')

print()
print('ALL CHECKS PASSED' if ok else 'SOME CHECKS FAILED')

# Fail the notebook loudly rather than leaving a silent FAIL line in the output.
assert ok, 'campaign consistency checks failed -- see the FAIL lines above'

### 4.3 Loading an L1 product with `corgidrp`

The products are ordinary Roman CGI L1 files, so the data reduction pipeline reads them
directly — the same path real Roman L1 data from MAST will follow.

> **Known `corgidrp` import-order issue.** `corgidrp/__init__.py` does not eagerly import its
> `check` submodule, so `corgidrp.data.Image(...)` raises
> `AttributeError: module 'corgidrp' has no attribute 'check'` unless `corgidrp.check` has been
> imported first. Importing it explicitly (below) is the workaround. This is a pre-existing
> `corgidrp` packaging quirk and says nothing about the validity of these L1 files.

In [ ]:
import corgidrp.check  # noqa: F401  -- must precede corgidrp.data.Image (see note above)
import corgidrp.data

example_path = l1_files[0]
image = corgidrp.data.Image(example_path)
print('loaded with corgidrp.data.Image:', os.path.basename(example_path))
print('  data shape / dtype :', image.data.shape, '/', image.data.dtype)
print('  VISITID / VISTYPE  :', image.pri_hdr['VISITID'], '/', image.pri_hdr['VISTYPE'])
print('  SATSPOTS / EXPTIME :', int(image.ext_hdr['SATSPOTS']), '/', image.ext_hdr['EXPTIME'])
print('  SCTSRT             :', image.ext_hdr['SCTSRT'])
del image  # keep no bulk frame data alive in the notebook

## 5. Locating the science crop on the full frame

The 153x153 simulated stamp is **not** at the centre of the 1200x2200 full frame. Two offsets
are involved:

1. `CorgiDetector.place_scene_on_detector` centres the stamp at `(loc_x, loc_y)` inside the
   **1024x1024 science area**.
2. That science area is embedded in the full frame at rows `13:1037` and columns `1088:2112`
   (the prescan/overscan layout).

**Mind the axis convention.** `place_scene_on_detector` performs

```python
full_frame[y_start:y_end, x_start:x_end] = sub_frame
```

so `loc_y` indexes **rows** (axis 0) and `loc_x` indexes **columns** (axis 1):

```
row    = 13   + loc_y
column = 1088 + loc_x
```

The default `LOC_X == LOC_Y == 512` makes both come out at row `525`, column `1600`, which
conveniently **hides** the reversal — so the mapping is written out explicitly below to avoid
silently transposing the crop as soon as a non-square placement is used.

> Getting this wrong is easy and quietly ruins any analysis: a naive "find the brightest pixel"
> search locks onto a **cosmic ray**, not the PSF. Always derive the crop from the geometry.

In [ ]:
SCIENCE_ROW0, SCIENCE_COL0 = 13, 1088          # science area origin in the full frame

# Axis convention: place_scene_on_detector assigns
#     full_frame[y_start:y_end, x_start:x_end] = sub_frame
# so loc_y indexes ROWS (axis 0) and loc_x indexes COLUMNS (axis 1).
# The default LOC_X == LOC_Y == 512 hides this, so keep the mapping explicit.
CROP_ROW = SCIENCE_ROW0 + LOC_Y
CROP_COL = SCIENCE_COL0 + LOC_X
HALF = OUTPUT_DIM // 2
CROP = (slice(CROP_ROW - HALF, CROP_ROW + HALF + 1),
        slice(CROP_COL - HALF, CROP_COL + HALF + 1))

print(f'science stamp centre: row {CROP_ROW}, column {CROP_COL}  (crop {OUTPUT_DIM}x{OUTPUT_DIM})')

# Load only the first trio's crops, so no bulk full-frame data is kept in the notebook.
trio_paths = l1_files[:3]
trio_labels = ['unprobed', 'positive', 'negative']
crops = []
for path in trio_paths:
    with fits.open(path, memmap=False) as hdul:
        crops.append(hdul[1].data[CROP].astype(float))
print('loaded crops:', [c.shape for c in crops])

## 6. Cosmic-ray-safe visualization

Every simulated EMCCD frame can contain **cosmic-ray hits** — isolated pixels of order 12,000 DN
against a background of roughly 175 DN. Consequently:

* **Never scale a display on `data.max()`.** A single cosmic ray compresses the entire
  astrophysical signal into the bottom fraction of the colour map.
* **Never summarise frames with `max()` or a plain RMS difference.** Both can be dominated
  by cosmic rays rather than by the probe.

Use **percentile limits** for display. The cell below scales every panel that way.

> **No quantitative signal numbers are reported here.** A simple brightness threshold does not
> reliably separate transient cosmic rays from legitimate optical signal — the probe itself
> produces bright, structured speckles — so any "masked mean" computed that way would be a
> heuristic, not a measurement. Quantitative probe photometry and normalized-intensity analysis
> belong to the deferred scientific validation step, which requires proper multi-frame cosmic-ray
> rejection and a defined dark-hole region.

In [ ]:
def percentile_limits(image, low=1.0, high=99.5):
    """Robust display limits: reduces sensitivity to isolated extreme pixels."""
    return np.percentile(image, low), np.percentile(image, high)

fig, axes = plt.subplots(1, 4, figsize=(17, 4.4))
vmin, vmax = percentile_limits(np.concatenate([c.ravel() for c in crops]))
for ax, image, label in zip(axes[:3], crops, trio_labels):
    im = ax.imshow(image, origin='lower', vmin=vmin, vmax=vmax, cmap='viridis')
    ax.set_title(f'{label}\n(percentile scaled)')
    fig.colorbar(im, ax=ax, fraction=0.046)

difference = crops[1] - crops[0]
dlim = np.percentile(np.abs(difference), 99.0)
im = axes[3].imshow(difference, origin='lower', vmin=-dlim, vmax=dlim, cmap='RdBu_r')
axes[3].set_title('positive - unprobed\n(symmetric percentile scale)')
fig.colorbar(im, ax=axes[3], fraction=0.046)
fig.suptitle(f'Alternate Probe trio, VISITID {summaries[0]["visitid"]} '
             f'(scale = {SCALE}, PROVISIONAL)')
fig.tight_layout()
plt.show()

# Display diagnostics only -- deliberately not a photometric measurement.
print(f'background (median)         : {np.median(np.array(crops)):.1f} DN')
print(f'display limits (1st/99.5th) : {vmin:.1f} / {vmax:.1f} DN')
print(f'raw maxima                  : ' + ', '.join(f'{c.max():.0f}' for c in crops)
      + '   <-- raw maxima may be cosmic-ray dominated; never use maxima as a probe metric.')

## 7. Summary and status

This notebook demonstrated the Alternate Probe L1 workflow end to end:

* the satellite-spot observing template is executed **once per custom probe**
  (`unprobed -> positive -> negative -> restore`), giving one `VISITID` per probe;
* custom Gaussian patterns are applied through the **existing** `add_satspot` / `remove_satspot`
  interface, with the analytical satellite-spot path untouched;
* L1 products are written by the **existing** detector and `outputs.save_hdu_to_fits` machinery,
  with no new FITS format and no new sequencing infrastructure;
* products are full-frame 1200x2200 `uint16`, load through `corgidrp.data`, and carry correct
  `SATSPOTS`, timestamps, `VISITID` and probe provenance.

### Still outstanding

* **Scientific scale / normalized-intensity validation is deferred**, pending discussion with
  Iva and Lukas. The `scale = 1.0` used throughout this notebook is a provisional
  demonstration value, **not** an approved science default, and no comparison against `0.3`
  and no NI measurement is performed here.
* `BANDPASS`, `VISTYPE`, the exposure/detector settings and the base DM solution remain
  **provisional infrastructure-test parameters** awaiting confirmation.
* The `corgidrp` `find_star` grouping convention (Phase 2) still has to be reconciled with the
  `SATSPOTS = 0/1` split produced here.

### Next phase

Develop the `corgidrp` recipe that processes these L1 datasets into the L2/L3 products needed
for the Alternate Probe analysis — the same path that real Roman L1 data from MAST will follow.